# 06 - Cross-Dataset Generalization & External Validation
### UROP: Improving Cross-Dataset Generalization and Severe-Stage Detection in ROP
This notebook investigates **Domain Shift (Problem 1)** through systematic cross-source evaluations:
- **Experiment A2**: Train on ROP-VL (China), zero-shot test on Ostrava (Czech Republic).
- **Experiment A3**: Train on Ostrava (Czech Republic), zero-shot test on ROP-VL (China).
- **Experiment A4**: Leave-One-Camera-Out on Ostrava (RetCam 3 vs Phoenix ICON vs RetCam Envision).
- **Experiment A5**: Zero-shot external benchmark evaluation on HVDROPDB (India).


In [1]:
import os, sys
if "COLAB_GPU" in os.environ or os.path.exists("/content"):
    sys.path.append("/content/drive/MyDrive/ROP_UROP")
    sys.path.append("/content/ROP_UROP")
else:
    sys.path.append(r"d:/UROP")

import torch
import pandas as pd
from src.models import get_model
from src.dataset import build_cross_dataset_dataloaders, build_leave_one_camera_out_dataloaders, build_external_test_dataloader
from src.losses import get_loss_function
from src.training import Trainer, set_seed
from configs import config

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
taskA_path = config.MANIFESTS_DIR / "taskA_manifest.csv"
taskA_df = pd.read_csv(taskA_path)


In [2]:
# Experiment A2: Train on ROP-VL -> Test on Ostrava (Zero-Shot Cross-Country Shift)
print("Setting up Experiment A2: ROP-VL (Train) -> Ostrava (Test)...")
train_loader_a2, test_loader_a2 = build_cross_dataset_dataloaders(
    taskA_df, train_source="ROPVL", test_source="OSTRAVA", batch_size=config.BATCH_SIZE
)
print(f"Train samples (ROP-VL): {len(train_loader_a2.dataset)} | Test samples (Ostrava): {len(test_loader_a2.dataset)}")


Setting up Experiment A2: ROP-VL (Train) -> Ostrava (Test)...
Train samples (ROP-VL): 1675 | Test samples (Ostrava): 4285


In [3]:
# Experiment A4: Leave-One-Camera-Out on Ostrava (Testing on Phoenix ICON)
print("Setting up Experiment A4: Ostrava Leave-One-Camera-Out (Testing Phoenix ICON)...")
train_loader_a4, test_loader_a4 = build_leave_one_camera_out_dataloaders(
    taskA_df, test_camera="Phoenix ICON", batch_size=config.BATCH_SIZE
)
print(f"Train samples (Other Cameras): {len(train_loader_a4.dataset)} | Test samples (Phoenix ICON): {len(test_loader_a4.dataset)}")


Setting up Experiment A4: Ostrava Leave-One-Camera-Out (Testing Phoenix ICON)...
Train samples (Other Cameras): 4285 | Test samples (Phoenix ICON): 0


In [4]:
# Experiment A5: Zero-Shot External Evaluation on HVDROPDB (India)
print("Setting up Experiment A5: External Evaluation on HVDROPDB...")
ext_loader = build_external_test_dataloader(taskA_df, external_source="HVDROPDB", batch_size=config.BATCH_SIZE)
print(f"External Benchmark Samples (HVDROPDB): {len(ext_loader.dataset)} (100 Normal, 85 ROP)")


Setting up Experiment A5: External Evaluation on HVDROPDB...
External Benchmark Samples (HVDROPDB): 185 (100 Normal, 85 ROP)
